## Dynamic World Analysis

In [1]:
import ee 
import geemap
ee.Authenticate()
ee.Initialize()

# Map Initialization
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

### Forest type (dry vs wet) layer for reprojection

In [2]:
# 10m resolution
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()

landcover = worldcover.select("Map")

forest = landcover.eq(10)

ecoregions = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017")

dry_ecoregion = ecoregions.filter(
    ee.Filter.eq("ECO_NAME", "Panamanian dry forests")
)

dry_mask = ee.Image.constant(0).paint(dry_ecoregion, 1).selfMask()

dry_forest = forest.updateMask(dry_mask)
wet_forest = forest.updateMask(dry_mask.unmask(0).Not())

classified = ee.Image(0)
classified = classified.where(wet_forest, 1)
classified = classified.where(dry_forest, 2)

Map.addLayer(
    classified.clip(panama_geom),
    {"min": 0, "max": 2, "palette": ["white", "006400", "d4a017"]},
    "Forest Type"
)

In [ ]:
classified.projection().getInfo()

{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}

In [4]:
classified.projection().nominalScale().getInfo()

111319.49079327357

### Dynamic World tested + reprojected

In [ ]:
# Construct a collection of corresponding Dynamic World and Sentinel-2 for
# inspection. Filter by region and date.
START = "2024-01-01"
END = "2024-12-31"

# 10m resolution
dw_col = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterBounds(panama_geom)
    .filterDate(START, END)
)

# TRY REPROJECTING DW_COL TO DRY VS WET DATASET PROJECTION

dw_image = ee.Image(dw_col.first())

# Create a visualization that blends DW class label with probability.
# Define list pairs of DW LULC label and color.
CLASS_NAMES = [
    'water',
    'trees',
    'grass',
    'flooded_vegetation',
    'crops',
    'shrub_and_scrub',
    'built',
    'bare',
    'snow_and_ice',
]

VIS_PALETTE = [
    '419bdf',
    '397d49',
    '88b053',
    '7a87c6',
    'e49635',
    'dfc35a',
    'c4281b',
    'a59b8f',
    'b39fe1',
]

# Create an RGB image of the label (most likely class) on [0, 1].
dw_rgb = (
    dw_image.select('label')
    .visualize(min=0, max=8, palette=VIS_PALETTE)
    .divide(255)
)

Map.addLayer(
    dw_rgb,
    {'min': 0, 'max': 0.65},
    'Dynamic World',
)
Map

Map(center=[8.5158389458998, -80.10966640141521], controls=(WidgetControl(options=['position', 'transparent_bg…

In [13]:
dw_col = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')#.filter(col_filter)

# linked_image = ee.Image(dw_col.first())

dw_col

In [ ]:

# Get the most likely class probability.
top1_prob = linked_image.select(CLASS_NAMES).reduce(ee.Reducer.max())

# Create a hillshade of the most likely class probability on [0, 1]
top1_prob_hillshade = ee.Terrain.hillshade(top1_prob.multiply(100)).divide(255)

# Combine the RGB image with the hillshade.
dw_rgb_hillshade = dw_rgb.multiply(top1_prob_hillshade)

# Display the Dynamic World visualization with the source Sentinel-2 image.
m = geemap.Map()
m.set_center(20.6729, 52.4305, 12)
m.add_layer(
    linked_image,
    {'min': 0, 'max': 3000, 'bands': ['B4', 'B3', 'B2']},
    'Sentinel-2 L1C',
)
m.add_layer(
    dw_rgb_hillshade,
    {'min': 0, 'max': 0.65},
    'Dynamic World V1 - label hillshade',
)
m